# CyberSec-FT-LLM — Unsloth QLoRA Fine-Tuning
Fine-tunes **Phi-3.5-mini-Instruct** on CVE/Exploit security data.

**Before running:**
1. Runtime → Change runtime type → **GPU (T4 or A100)**
2. Upload `dataset/` folder to: `My Drive/CyberSec-FT-LLM/dataset/`
3. Run all cells in order ▶

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q unsloth trl>=0.9.0 transformers>=4.45.0 datasets accelerate peft bitsandbytes pyyaml rouge-score
print('✅ Dependencies installed.')

In [ ]:
# ── Cell 2: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA : {torch.cuda.is_available()}')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 3: Mount Drive & Verify Dataset ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
DATASET_DIR = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    p = os.path.join(DATASET_DIR, f)
    print(f'  {f}: {os.path.getsize(p)/1e6:.1f} MB' if os.path.exists(p) else f'  {f}: ❌ NOT FOUND!')

In [ ]:
# ── Cell 4: Clone Repo ────────────────────────────────────────────────────────
import os
REPO_URL = 'https://github.com/Mohamedabul/CyberSec-FT-LLM.git'
REPO_DIR = '/content/CyberSec-FT-LLM'
if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 5: Link Dataset from Drive ──────────────────────────────────────────
import os
SRC = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
DST = '/content/CyberSec-FT-LLM/dataset'
os.makedirs(DST, exist_ok=True)
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    dst = os.path.join(DST, f)
    if os.path.exists(dst) or os.path.islink(dst): os.remove(dst)
    src = os.path.join(SRC, f)
    if os.path.exists(src):
        os.symlink(src, dst); print(f'  ✅ Linked: {f}')
    else:
        print(f'  ❌ MISSING: {f} — upload to Drive first!')

In [ ]:
# ── Cell 6: Configure Colab Paths ────────────────────────────────────────────
import yaml, os
config_path = '/content/CyberSec-FT-LLM/configs/training_config.yaml'
if not os.path.exists(config_path):
    raise FileNotFoundError(f'Not found: {config_path} — did Cell 4 complete?')
with open(config_path) as f:
    cfg_yaml = yaml.safe_load(f)
cfg_yaml['colab']['output_dir']   = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
cfg_yaml['colab']['dataset_path'] = '/content/CyberSec-FT-LLM/dataset'
with open(config_path, 'w') as f:
    yaml.dump(cfg_yaml, f, default_flow_style=False)
os.makedirs(cfg_yaml['colab']['output_dir'], exist_ok=True)
print('✅ Config updated:')
print(f"  output_dir  : {cfg_yaml['colab']['output_dir']}")
print(f"  dataset_path: {cfg_yaml['colab']['dataset_path']}")

---
## Training — Cells 7a through 7e
Each step is a separate cell so you can re-run individual steps if needed.

| Cell | Step | What you'll see |
|------|------|-----------------|
| 7a | Load Model | Shard loading progress bar |
| 7b | Load Dataset | Tokenization progress bar |
| 7c | Configure Trainer | Hyperparams summary |
| 7d | **Train** | Live: `loss / grad_norm / lr / epoch` every 25 steps. Checkpoint saved to Drive every 200 steps. |
| 7e | Save + Merge | Adapter save + full merged model on Drive |

In [ ]:
# ── Cell 7a: Load Model (4-bit Unsloth) ──────────────────────────────────────
import yaml, sys
from pathlib import Path
sys.path.insert(0, '/content/CyberSec-FT-LLM/training/unsloth')
from model_loader_unsloth import load_model_and_tokenizer

BASE = Path('/content/CyberSec-FT-LLM')
model_cfg = yaml.safe_load(open(BASE/'configs'/'model_config.yaml'))
train_cfg = yaml.safe_load(open(BASE/'configs'/'training_config.yaml'))
tc, cc = train_cfg['common'], train_cfg['colab']

cfg = {
    'model':        {**model_cfg['base_model'], 'max_seq_length': model_cfg['tokenizer']['max_length']},
    'quantization': model_cfg['quantization'],
    'lora':         model_cfg['lora'],
}

print('Loading Phi-3.5-mini-Instruct in 4-bit (Unsloth)...')
model, tokenizer = load_model_and_tokenizer(cfg)
print('✅ Model ready.')

In [ ]:
# ── Cell 7b: Load & Format Datasets ─────────────────────────────────────────
import json
from datasets import Dataset
from pathlib import Path

DATASET_BASE = Path(cc['dataset_path'])

def load_jsonl(path):
    rows = []
    for line in open(path, encoding='utf-8', errors='replace'):
        line = line.replace('\x00','').strip()
        if line:
            try: rows.append(json.loads(line))
            except: pass
    return rows

def fmt(s):
    u = f"{s.get('instruction','')}\n\n{s.get('input','')}" if s.get('input') else s.get('instruction','')
    return tokenizer.apply_chat_template(
        [{'role':'user','content':u}, {'role':'assistant','content':s.get('output','')}],
        tokenize=False, add_generation_prompt=False)

print('Loading train.jsonl...')
train_raw = load_jsonl(str(DATASET_BASE/'train.jsonl'))
print(f'  {len(train_raw):,} train samples')

print('Loading val.jsonl...')
val_raw = load_jsonl(str(DATASET_BASE/'val.jsonl'))
print(f'  {len(val_raw):,} val samples')

print('Formatting...')
train_ds = Dataset.from_list([{'text': fmt(s)} for s in train_raw])
val_ds   = Dataset.from_list([{'text': fmt(s)} for s in val_raw])
del train_raw, val_raw
print(f'✅ Datasets ready — Train: {len(train_ds):,} | Val: {len(val_ds):,}')

In [ ]:
# ── Cell 7c: Configure Trainer ────────────────────────────────────────────────
import os, glob
from trl import SFTTrainer, SFTConfig

adapter_dir = cc['output_dir']
os.makedirs(adapter_dir, exist_ok=True)

args = SFTConfig(
    output_dir=adapter_dir,
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=cc['per_device_train_batch_size'],
    gradient_accumulation_steps=cc['gradient_accumulation_steps'],
    learning_rate=tc['learning_rate'],
    lr_scheduler_type=tc['lr_scheduler_type'],
    warmup_steps=tc['warmup_steps'],
    weight_decay=tc['weight_decay'],
    bf16=cc['bf16'],
    gradient_checkpointing=tc['gradient_checkpointing'],
    optim=cc['optim'],
    logging_steps=tc['logging_steps'],
    save_steps=tc['save_steps'],
    eval_steps=tc['eval_steps'],
    eval_strategy='steps',
    save_total_limit=tc['save_total_limit'],
    load_best_model_at_end=tc['load_best_model_at_end'],
    report_to=tc['report_to'],
    seed=tc['seed'],
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds, args=args,
)

print('✅ Trainer configured:')
print(f'  Epochs           : {tc["num_train_epochs"]}')
print(f'  Batch size        : {cc["per_device_train_batch_size"]} x {cc["gradient_accumulation_steps"]} acc = {cc["per_device_train_batch_size"]*cc["gradient_accumulation_steps"]} effective')
print(f'  Learning rate     : {tc["learning_rate"]}')
print(f'  Optimizer         : {cc["optim"]}')
print(f'  Checkpoint every  : {tc["save_steps"]} steps → {adapter_dir}')
print(f'  Log every         : {tc["logging_steps"]} steps')

In [ ]:
# ── Cell 7d: Train ────────────────────────────────────────────────────────────
# Live output every 25 steps:
#   {'loss': 1.43, 'grad_norm': 0.85, 'learning_rate': 0.00019, 'epoch': 0.01}
# Checkpoint saved to Drive every 200 steps.
# Re-run this cell anytime to auto-resume from last checkpoint.

checkpoints = sorted(glob.glob(os.path.join(adapter_dir, 'checkpoint-*')))
resume_from = checkpoints[-1] if checkpoints else None

if resume_from:
    print(f'🔁 Resuming from: {os.path.basename(resume_from)}')
else:
    print('🚀 Starting fresh training...')

trainer.train(resume_from_checkpoint=resume_from)
print('\n✅ Training complete!')

In [ ]:
# ── Cell 7e: Save Adapter + Merge Full Model ──────────────────────────────────
import os
from unsloth import FastLanguageModel

# Save LoRA adapter
print('Saving LoRA adapter...')
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'  ✅ Adapter saved to: {adapter_dir}')

# Merge LoRA into base model (full 16-bit weights)
print('\nMerging LoRA into base model (this takes a few minutes)...')
merged_dir = adapter_dir.replace('/adapter', '/merged')
merged, mtok = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=cfg['model']['max_seq_length'],
    dtype=None, load_in_4bit=True,
)
os.makedirs(merged_dir, exist_ok=True)
merged.save_pretrained_merged(merged_dir, mtok, save_method='merged_16bit')
print(f'  ✅ Merged model saved to: {merged_dir}')
print('\n🎉 All done! Both adapter and merged model saved to Google Drive.')

In [ ]:
# ── Cell 8: Quick Inference Test ──────────────────────────────────────────────
import sys, torch
sys.path.insert(0, '/content/CyberSec-FT-LLM/training/unsloth')
from model_loader_unsloth import load_for_inference

ADAPTER = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
model_inf, tok_inf = load_for_inference(ADAPTER, max_seq_length=512)

instruction = 'Analyze the following CVE and provide a structured vulnerability report.'
context = (
    'CVE ID: CVE-2021-44228\n'
    'Description: Apache Log4j2 JNDI RCE vulnerability (Log4Shell).\n'
    'CVSS v3: 10.0 CRITICAL | Attack Vector: NETWORK | CWE: CWE-917'
)
msgs = [{'role': 'user', 'content': f'{instruction}\n\n{context}'}]
prompt = tok_inf.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok_inf(prompt, return_tensors='pt').to(model_inf.device)
with torch.no_grad():
    out = model_inf.generate(**inputs, max_new_tokens=400, temperature=0.7, do_sample=True)
print(tok_inf.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# ── Cell 9: Verify Saved Files on Drive ──────────────────────────────────────
import os
for folder in [
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter',
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/merged',
]:
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if not os.path.isdir(os.path.join(folder, f))]
        size  = sum(os.path.getsize(os.path.join(folder, f)) for f in files) / 1e6
        print(f'✅ {folder}\n   {len(files)} files | {size:.0f} MB')
        for f in sorted(files): print(f'     {f}')
    else:
        print(f'❌ Not found: {folder}')